In [4]:
import os, json, math, time, re
from pathlib import Path
from typing import List, Tuple

import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModel, AutoConfig

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

PROMPT_TEMPLATE = """<image>
Analyze this image carefully. Determine if a person has fallen down.

Important classification rules:
- The "falldown" category applies to any person who is lying down, regardless of:
  - the surface (e.g., floor, mattress, bed)
  - the posture (natural or unnatural)
  - the cause (e.g., sleeping, collapsing, lying intentionally)
- This includes:
  - A person lying flat on the ground or other surfaces
  - A person collapsed or sprawled in any lying position
- The "normal" category applies only if the person is:
  - sitting
  - standing
  - kneeling
  - or otherwise upright (not lying down)

Answer in JSON format with BOTH of the following fields:
- "category": either "falldown" or "normal"
- "description": a brief reason why this classification was made (e.g., "person lying on a mattress", "person sitting on sofa")
"""


import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

def build_transform(input_size: int = 448):
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float("inf")
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        tgt_ar = ratio[0] / ratio[1]
        diff = abs(aspect_ratio - tgt_ar)
        if diff < best_ratio_diff or (diff == best_ratio_diff and area > 0.5 * image_size * image_size * ratio[0] * ratio[1]):
            best_ratio_diff = diff
            best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=True):
    ow, oh = image.size
    aspect_ratio = ow / oh
    target_ratios = sorted(
        {(i, j) for n in range(min_num, max_num + 1)
         for i in range(1, n + 1) for j in range(1, n + 1)
         if min_num <= i * j <= max_num},
        key=lambda x: x[0] * x[1],
    )
    ratio = find_closest_aspect_ratio(aspect_ratio, target_ratios, ow, oh, image_size)
    tw, th = image_size * ratio[0], image_size * ratio[1]
    blocks = ratio[0] * ratio[1]
    resized = image.resize((tw, th))
    tiles = []
    for idx in range(blocks):
        x = (idx % (tw // image_size)) * image_size
        y = (idx // (tw // image_size)) * image_size
        tiles.append(resized.crop((x, y, x + image_size, y + image_size)))
    if use_thumbnail and blocks != 1:
        tiles.append(image.resize((image_size, image_size)))
    return tiles

def parse_prediction(pred_str: str) -> str:
    try:
        clean = pred_str
        if '```json' in clean:
            clean = clean.split('```json')[1].split('```')[0]
        elif '```' in clean:
            clean = clean.split('```')[1].split('```')[0]
        clean = clean.strip()
        s, e = clean.find('{'), clean.rfind('}')
        if s != -1 and e != -1 and s < e:
            json_part = clean[s:e+1]
            try:
                data = json.loads(json_part)
                cat = data.get('category')
                if cat in ['falldown', 'normal']:
                    return cat
            except json.JSONDecodeError:
                pass
        m = re.search(r'["\']category["\']\s*:\s*["\'](falldown|normal)["\']', clean)
        if m:
            return m.group(1)
        return 'no_json_found'
    except Exception:
        return 'parsing_failed'
def load_image(image_file, input_size=448, max_num=12):
    image = Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values

checkpoint = "ckpts/InternVL3-2B_hyundai_5_20"

model = AutoModel.from_pretrained(
    checkpoint, torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True, trust_remote_code=True,
).eval().cuda()
tokenizer = AutoTokenizer.from_pretrained(checkpoint, trust_remote_code=True, use_fast=False)
image_size = model.config.force_image_size or model.config.vision_config.image_size

pixel_values = load_image('5.png', max_num=12).to(torch.bfloat16).cuda()
generation_config = dict(max_new_tokens=1024, do_sample=False)
response = model.chat(tokenizer, pixel_values, PROMPT_TEMPLATE , generation_config)
print(f'Assistant: {response}')


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Assistant: {"category": "falldown", "description": "In an industrial recycling facility with a concrete floor, a man is lying in the middle of the aisle. He is wearing a grey jumpsuit and is positioned on his back, with his torso completely horizontal to the ground, indicating a clear fall-down event."}
